In [1]:
%pip install pypdf chromadb openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#Loading and reading a markdown file

from pathlib import Path

markdown_path = Path("documents/ai_notes.md")

with open(markdown_path,"r",encoding="utf-8") as file:
    markdown_text=file.read()

print(markdown_text)

# Artificial Intelligence Notes

## Machine Learning
Machine learning allows computers to learn patterns
from data without being explicitly programmed for
every individual task.

## Deep Learning
Deep learning is a subset of machine learning that
uses neural networks with multiple layers.

## Retrieval Augmented Generation
RAG combines document retrieval with language
generation. It retrieves relevant information from
external documents before generating an answer.

## Embeddings
Embeddings represent text as numerical vectors.
They help computers compare the meaning of text.


In [3]:
#Loading and reading a PDF file
from pypdf import PdfReader

pdf_path = Path("documents/sample_notes.pdf")

reader = PdfReader(pdf_path)

pdf_text =""

for page_number,page in enumerate(reader.pages,start=1):
    page_text=page.extract_text() or ""
    pdf_text +=f"\n\n{page_text}"
print(pdf_text)



 Introduction to Artificial Intelligence
Artificial Intelligence (AI) refers to the ability of machines to perform tasks that typically require human
intelligence. These tasks include learning, reasoning, problem-solving, perception, and language
understanding. AI systems are increasingly used in healthcare, finance, education, and transportation.
Types of AI
Narrow AI
Narrow AI is designed to perform a specific task. Examples include chatbots, recommendation systems,
and voice assistants such as Siri and Alexa.
General AI
General AI refers to a hypothetical system capable of performing any intellectual task that a human can
do. General AI has not yet been achieved.
Machine Learning
Machine Learning (ML) is a subset of AI that enables systems to learn from data without being explicitly
programmed. Instead of following fixed rules, ML models identify patterns in historical data and use
those patterns to make predictions.
Supervised Learning
In supervised learning, models are trained u

In [4]:
def create_chunks(text, chunk_size=500, overlap=50):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [5]:
markdown_chunks = create_chunks(
    markdown_text,
    chunk_size=500,
    overlap=50
)

print("Total chunks:", len(markdown_chunks))

for i, chunk in enumerate(markdown_chunks):
    print(f"\nCHUNK {i + 1}")
    print(chunk)

Total chunks: 2

CHUNK 1
# Artificial Intelligence Notes

## Machine Learning
Machine learning allows computers to learn patterns
from data without being explicitly programmed for
every individual task.

## Deep Learning
Deep learning is a subset of machine learning that
uses neural networks with multiple layers.

## Retrieval Augmented Generation
RAG combines document retrieval with language
generation. It retrieves relevant information from
external documents before generating an answer.

## Embeddings
Embeddings repr

CHUNK 2
nerating an answer.

## Embeddings
Embeddings represent text as numerical vectors.
They help computers compare the meaning of text.


In [6]:
#For pdf we'll preserve the roll number separately

pdf_chunks=[]

for page_number,page in enumerate(reader.pages,start=1):
    page_text=page.extract_text() or ""
    if page_text.strip():
        page_chunks=create_chunks(
            page_text,
            chunk_size=500,
            overlap=50
        )

        for chunk_number,chunk in enumerate(page_chunks,start=1):
            pdf_chunks.append({
                "text":chunk,
                "source":"sample_notes.pdf",
                "page":page_number,
                "chunk":chunk_number
            })
print("Total PDF chunks:",len(pdf_chunks))

Total PDF chunks: 9


In [7]:
print(pdf_chunks)

[{'text': ' Introduction to Artificial Intelligence\nArtificial Intelligence (AI) refers to the ability of machines to perform tasks that typically require human\nintelligence. These tasks include learning, reasoning, problem-solving, perception, and language\nunderstanding. AI systems are increasingly used in healthcare, finance, education, and transportation.\nTypes of AI\nNarrow AI\nNarrow AI is designed to perform a specific task. Examples include chatbots, recommendation systems,\nand voice assistants such a', 'source': 'sample_notes.pdf', 'page': 1, 'chunk': 1}, {'text': 'ecommendation systems,\nand voice assistants such as Siri and Alexa.\nGeneral AI\nGeneral AI refers to a hypothetical system capable of performing any intellectual task that a human can\ndo. General AI has not yet been achieved.\nMachine Learning\nMachine Learning (ML) is a subset of AI that enables systems to learn from data without being explicitly\nprogrammed. Instead of following fixed rules, ML models ident

In [10]:
#Storing document chunks in ChromaDB

import os
import chromadb

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client=OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

chroma_client=chromadb.PersistentClient(
    path="./chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="day20_documents"
)

In [11]:
all_documents = []
all_metadata = []
all_ids = []

for i, chunk in enumerate(markdown_chunks):
    all_documents.append(chunk)

    all_metadata.append({
        "source": "ai_notes.md",
        "page": 0,
        "file_type": "markdown"
    })

    all_ids.append(f"markdown_{i}")

for item in pdf_chunks:
    all_documents.append(item["text"])

    all_metadata.append({
        "source": item["source"],
        "page": item["page"],
        "chunk": item["chunk"],
        "file_type": "pdf"
    })

    all_ids.append(
        f"pdf_page_{item['page']}_chunk_{item['chunk']}"
    )

print("Total chunks:", len(all_documents))

Total chunks: 11


In [12]:
#Generating Embeddings
embedding_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=all_documents
)

embeddings = [
    item.embedding
    for item in embedding_response.data
]

print("Embeddings generated:", len(embeddings))

Embeddings generated: 11


In [13]:
#Finally saving them in ChromaDB

collection.upsert(
    ids=all_ids,
    documents=all_documents,
    embeddings=embeddings,
    metadatas=all_metadata
)

print("Stored chunks:", collection.count())

Stored chunks: 11


In [15]:
#Testing document retrieval
question = "What is retrieval augmented generation?"

#Generating the query embedding
query_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=question
)

query_embedding = query_response.data[0].embedding

#Search ChromaDB
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [16]:
#display the results with source information
for i, document in enumerate(results["documents"][0]):
    metadata = results["metadatas"][0][i]

    print("\nRetrieved document:")
    print(document)

    print("Source:", metadata["source"])
    print("Page:", metadata["page"])


Retrieved document:
# Artificial Intelligence Notes

## Machine Learning
Machine learning allows computers to learn patterns
from data without being explicitly programmed for
every individual task.

## Deep Learning
Deep learning is a subset of machine learning that
uses neural networks with multiple layers.

## Retrieval Augmented Generation
RAG combines document retrieval with language
generation. It retrieves relevant information from
external documents before generating an answer.

## Embeddings
Embeddings repr
Source: ai_notes.md
Page: 0

Retrieved document:
ses store embeddings and perform similarity searches efficiently.
Popular vector databases include:
• ChromaDB 
• Pinecone 
• Weaviate 
• Milvus 
• Qdrant 
These databases are commonly used in Retrieval-Augmented Generation (RAG) applications.
Retrieval-Augmented Generation
RAG combines information retrieval with large language models.
Workflow:
1. Load documents. 
2. Split documents into chunks. 
3. Convert chunks into embed